### Load CPEC glacial lake data 

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import re
from sklearn.neighbors import BallTree

gpkg_path = '../data/raw/cpec_glaciallake.gpkg'

gdf_1990 = gpd.read_file(
    gpkg_path,
    layer='cpec_glaciallake_1990_landsat'
)

gdf_2000 = gpd.read_file(
    gpkg_path,
    layer='cpec_glaciallake_2000_landsat'
)

gdf_2020 = gpd.read_file(
    gpkg_path,
    layer='cpec_glaciallake_2020_landsat'
)

print("1990 lakes:", len(gdf_1990))
print("2000 lakes:", len(gdf_2000))
print("2020 lakes:", len(gdf_2020))

1990 lakes: 2154
2000 lakes: 2184
2020 lakes: 2234


### Convert coordinates from DMS text format to decimal degrees

In [3]:
def dms_to_decimal(dms_str):
    if pd.isna(dms_str):
        return np.nan

    match = re.match(
        r"(\d+)°\s*(\d+)'\s*([\d.]+)\"\s*([NSEW])",
        str(dms_str).strip()
    )

    if not match:
        return np.nan

    degrees, minutes, seconds, direction = match.groups()

    decimal = (
        float(degrees)
        + float(minutes) / 60
        + float(seconds) / 3600
    )

    if direction in ['S', 'W']:
        decimal = -decimal

    return decimal


for gdf in [gdf_1990, gdf_2000, gdf_2020]:
    gdf['latitude_decimal'] = gdf['Latitude'].apply(dms_to_decimal)
    gdf['longitude_decimal'] = gdf['Longitude'].apply(dms_to_decimal)

print("Failed 1990:",
      gdf_1990[['latitude_decimal', 'longitude_decimal']].isna().any(axis=1).sum())

print("Failed 2000:",
      gdf_2000[['latitude_decimal', 'longitude_decimal']].isna().any(axis=1).sum())

print("Failed 2020:",
      gdf_2020[['latitude_decimal', 'longitude_decimal']].isna().any(axis=1).sum())

Failed 1990: 0
Failed 2000: 0
Failed 2020: 0


### Convert area from square meters to square kilometers

In [4]:
for gdf in [gdf_1990, gdf_2000, gdf_2020]:
    gdf['area_km2'] = gdf['Area'] / 1_000_000

print("1990 area:")
print(gdf_1990['area_km2'].describe())

print("\n2000 area:")
print(gdf_2000['area_km2'].describe())

print("\n2020 area:")
print(gdf_2020['area_km2'].describe())

1990 area:
count    2154.000000
mean        0.039507
std         0.138538
min         0.004500
25%         0.009000
50%         0.017100
75%         0.035100
max         4.860000
Name: area_km2, dtype: float64

2000 area:
count    2184.000000
mean        0.039421
std         0.138874
min         0.004500
25%         0.009000
50%         0.016200
75%         0.035100
max         4.860000
Name: area_km2, dtype: float64

2020 area:
count    2234.000000
mean        0.038634
std         0.136427
min         0.004500
25%         0.009000
50%         0.016200
75%         0.035100
max         4.860000
Name: area_km2, dtype: float64


### Check for missing, zero, or negative area values

In [5]:
for year, gdf in [(1990, gdf_1990), (2000, gdf_2000), (2020, gdf_2020)]:
    print(
        year,
        "missing area:", gdf['area_km2'].isna().sum(),
        "| zero/negative area:", (gdf['area_km2'] <= 0).sum()
    )

1990 missing area: 0 | zero/negative area: 0
2000 missing area: 0 | zero/negative area: 0
2020 missing area: 0 | zero/negative area: 0


### Spatially match 1990 lakes to 2000 lakes and check matching distances

In [6]:
# Prepare coordinates for spatial matching

coords_1990 = np.radians(
    gdf_1990[['latitude_decimal', 'longitude_decimal']].values
)

coords_2000 = np.radians(
    gdf_2000[['latitude_decimal', 'longitude_decimal']].values
)

# Build BallTree using Haversine distance
tree_2000 = BallTree(
    coords_2000,
    metric='haversine'
)

# Find the closest 2000 lake for every 1990 lake
distances_1990_2000, indices_1990_2000 = tree_2000.query(
    coords_1990,
    k=1
)

# Convert radians to kilometers
distance_km_1990_2000 = (
    distances_1990_2000.flatten() * 6371
)

print("1990 → 2000 matching distances:")
print(pd.Series(distance_km_1990_2000).describe())

1990 → 2000 matching distances:
count    2154.000000
mean        0.293121
std         1.503694
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        19.285693
dtype: float64


In [7]:
# Check percentile breakdown more granularly
for p in [50, 75, 90, 95, 98, 99, 99.5, 100]:
    print(f"{p}th percentile: {np.percentile(distance_km_1990_2000, p):.4f} km")

print()
print("Exact zero matches:", (distance_km_1990_2000 == 0).sum())
print("Matches under 0.5 km:", (distance_km_1990_2000 < 0.5).sum())
print("Matches under 1 km:", (distance_km_1990_2000 < 1).sum())
print("Matches over 5 km:", (distance_km_1990_2000 > 5).sum())

50th percentile: 0.0000 km
75th percentile: 0.0000 km
90th percentile: 0.3566 km
95th percentile: 1.2281 km
98th percentile: 3.7407 km
99th percentile: 6.1412 km
99.5th percentile: 12.8731 km
100th percentile: 19.2857 km

Exact zero matches: 1690
Matches under 0.5 km: 1970
Matches under 1 km: 2029
Matches over 5 km: 28


In [8]:
MATCH_THRESHOLD_KM = 1.0

# Flag which 1990 lakes have a reliable match
reliable_match = distance_km_1990_2000 <= MATCH_THRESHOLD_KM

print(f"Reliable matches (≤{MATCH_THRESHOLD_KM} km): {reliable_match.sum()}")
print(f"Unreliable/no match (>{MATCH_THRESHOLD_KM} km): {(~reliable_match).sum()}")
print(f"Percentage reliable: {reliable_match.mean()*100:.1f}%")

Reliable matches (≤1.0 km): 2029
Unreliable/no match (>1.0 km): 125
Percentage reliable: 94.2%


### Spatially match 2000 lakes to 2020 lakes and check matching distances

In [9]:
# Prepare coordinates for spatial matching

coords_2000_for_2020 = np.radians(
    gdf_2000[['latitude_decimal', 'longitude_decimal']].values
    )

coords_2020 = np.radians(
    gdf_2020[['latitude_decimal', 'longitude_decimal']].values
    )

# Build BallTree using Haversine distance
tree_2020 = BallTree(
    coords_2020, 
    metric='haversine'
    )

# Find the closest 2000 lake for every 1990 lake
distances_2000_2020, indices_2000_2020 = tree_2020.query(
    coords_2000_for_2020, 
    k=1
    )

# Convert radians to kilometers
distance_km_2000_2020 = (
    distances_2000_2020.flatten() * 6371
    )

print("2000 → 2020 matching distances:")
print(pd.Series(distance_km_2000_2020).describe())

print()
for p in [50, 75, 90, 95, 98, 99, 100]:
    print(f"{p}th percentile: {np.percentile(distance_km_2000_2020, p):.4f} km")

2000 → 2020 matching distances:
count    2184.000000
mean        0.174524
std         0.778640
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        13.759733
dtype: float64

50th percentile: 0.0000 km
75th percentile: 0.0000 km
90th percentile: 0.3581 km
95th percentile: 0.9955 km
98th percentile: 2.2471 km
99th percentile: 3.1676 km
100th percentile: 13.7597 km


### Apply reliability threshold to 2000→2020 matches

In [10]:
reliable_match_2000_2020 = distance_km_2000_2020 <= MATCH_THRESHOLD_KM

print(f"Reliable matches (≤{MATCH_THRESHOLD_KM} km): {reliable_match_2000_2020.sum()}")
print(f"Unreliable/no match (>{MATCH_THRESHOLD_KM} km): {(~reliable_match_2000_2020).sum()}")
print(f"Percentage reliable: {reliable_match_2000_2020.mean()*100:.1f}%")

Reliable matches (≤1.0 km): 2075
Unreliable/no match (>1.0 km): 109
Percentage reliable: 95.0%


### Build the full 1990-2000-2020 chain

In [11]:
# Build a table linking 1990 -> 2000 -> 2020 using the match indices we already computed

chain = pd.DataFrame({
    'idx_1990': range(len(gdf_1990)),
    'idx_2000': indices_1990_2000.flatten(),
    'reliable_1990_2000': reliable_match
})

# For each matched 2000 lake, find its corresponding 2020 match
chain['idx_2020'] = chain['idx_2000'].map(
    lambda i: indices_2000_2020.flatten()[i]
)
chain['reliable_2000_2020'] = chain['idx_2000'].map(
    lambda i: reliable_match_2000_2020[i]
)

# Keep only lakes reliable at BOTH stages
chain['fully_reliable'] = chain['reliable_1990_2000'] & chain['reliable_2000_2020']

print("Total 1990 lakes:", len(chain))
print("Fully reliable chain (1990->2000->2020):", chain['fully_reliable'].sum())
print("Percentage fully reliable:", chain['fully_reliable'].mean()*100, "%")

Total 1990 lakes: 2154
Fully reliable chain (1990->2000->2020): 1972
Percentage fully reliable: 91.55060352831941 %


### Calculate growth rate metrics for lakes with a fully reliable 1990-2000-2020 match

In [12]:
# Filter to only the fully reliable lakes
reliable_chain = chain[chain['fully_reliable']].copy()

# Pull the area values for each matched lake across the three years
reliable_chain['area_1990'] = gdf_1990.iloc[reliable_chain['idx_1990']]['area_km2'].values
reliable_chain['area_2000'] = gdf_2000.iloc[reliable_chain['idx_2000']]['area_km2'].values
reliable_chain['area_2020'] = gdf_2020.iloc[reliable_chain['idx_2020']]['area_km2'].values

# Also carry over the 2020 coordinates - this is what we'll use to merge with your main dataset later
reliable_chain['latitude'] = gdf_2020.iloc[reliable_chain['idx_2020']]['latitude_decimal'].values
reliable_chain['longitude'] = gdf_2020.iloc[reliable_chain['idx_2020']]['longitude_decimal'].values

# Calculate growth metrics
reliable_chain['area_change_km2'] = reliable_chain['area_2020'] - reliable_chain['area_1990']
reliable_chain['annual_area_change_km2_per_year'] = reliable_chain['area_change_km2'] / 30
reliable_chain['growth_rate_percent'] = (
    (reliable_chain['area_2020'] - reliable_chain['area_1990']) / reliable_chain['area_1990']
) * 100

print("Growth rate summary:")
print(reliable_chain[['area_1990', 'area_2000', 'area_2020', 'area_change_km2', 
                        'annual_area_change_km2_per_year', 'growth_rate_percent']].describe())

Growth rate summary:
         area_1990    area_2000    area_2020  area_change_km2  \
count  1972.000000  1972.000000  1972.000000      1972.000000   
mean      0.041874     0.042034     0.042509         0.000635   
std       0.144471     0.145156     0.145186         0.018192   
min       0.004500     0.004500     0.004500        -0.280800   
25%       0.009900     0.009900     0.009900         0.000000   
50%       0.018000     0.018000     0.018000         0.000000   
75%       0.037125     0.037800     0.037800         0.000000   
max       4.860000     4.860000     4.860000         0.401400   

       annual_area_change_km2_per_year  growth_rate_percent  
count                      1972.000000          1972.000000  
mean                          0.000021            12.355005  
std                           0.000606           142.011591  
min                          -0.009360           -94.791667  
25%                           0.000000             0.000000  
50%                  

In [13]:
# Check how many lakes have these extreme values
print("Lakes with growth_rate_percent > 500%:", (reliable_chain['growth_rate_percent'] > 500).sum())
print("Lakes with growth_rate_percent < -80%:", (reliable_chain['growth_rate_percent'] < -80).sum())

# Look at what's driving the extreme cases
extreme = reliable_chain[
    (reliable_chain['growth_rate_percent'] > 500) | (reliable_chain['growth_rate_percent'] < -80)
]
print(extreme[['area_1990', 'area_2000', 'area_2020', 'growth_rate_percent']])

Lakes with growth_rate_percent > 500%: 12
Lakes with growth_rate_percent < -80%: 13
      area_1990  area_2000  area_2020  growth_rate_percent
291      0.0243   0.007200     0.0045           -81.481481
854      0.0045   0.031500     0.0612          1260.000000
855      0.0045   0.031500     0.0612          1260.000000
858      0.0045   0.018000     0.0306           580.000000
1269     0.0540   0.004500     0.4095           658.333333
1270     0.0945   0.004500     0.0099           -89.523810
1271     0.0081   0.010800     0.4095          4955.555556
1512     0.0432   0.043200     0.0063           -85.416667
1513     0.1449   0.144900     0.0135           -90.683230
1537     0.0045   0.009900     0.0621          1280.000000
1538     0.0045   0.009900     0.0621          1280.000000
1539     0.0081   0.015300     0.0621           666.666667
1577     0.0045   0.065700     0.0657          1360.000000
1589     0.0054   0.010800     0.0432           700.000000
1596     0.0045   0.010419     

In [14]:
print("Final feature to use: annual_area_change_km2_per_year")
print(reliable_chain['annual_area_change_km2_per_year'].describe())

Final feature to use: annual_area_change_km2_per_year
count    1972.000000
mean        0.000021
std         0.000606
min        -0.009360
25%         0.000000
50%         0.000000
75%         0.000000
max         0.013380
Name: annual_area_change_km2_per_year, dtype: float64


In [15]:
# Check whether multiple 1990 lakes were assigned to the same 2000 lake
print("Unique 2000 matches:", reliable_chain['idx_2000'].nunique())
print("Total reliable chains:", len(reliable_chain))

# Check whether multiple chains were assigned to the same 2020 lake
print("Unique 2020 matches:", reliable_chain['idx_2020'].nunique())
print("Total reliable chains:", len(reliable_chain))

# Check duplicates
print("\nDuplicate 2000 assignments:",
      reliable_chain['idx_2000'].duplicated().sum())

print("Duplicate 2020 assignments:",
      reliable_chain['idx_2020'].duplicated().sum())

Unique 2000 matches: 1903
Total reliable chains: 1972
Unique 2020 matches: 1876
Total reliable chains: 1972

Duplicate 2000 assignments: 69
Duplicate 2020 assignments: 96


### Inspect 2000 lakes that were assigned to multiple 1990 lakes

In [16]:
# Inspect 2000 lakes that were assigned to multiple 1990 lakes

duplicate_2000 = reliable_chain[
    reliable_chain['idx_2000'].duplicated(keep=False)
].sort_values('idx_2000')

print("Duplicate 2000 assignments:")
print(
    duplicate_2000[
        ['idx_1990', 'idx_2000', 'reliable_1990_2000']
    ].to_string(index=False)
)

print("\nNumber of duplicated 2000 lakes:",
      duplicate_2000['idx_2000'].nunique())

Duplicate 2000 assignments:
 idx_1990  idx_2000  reliable_1990_2000
      287         0                True
      289         0                True
      290         0                True
     1847         6                True
     1846         6                True
     1550       207                True
     1549       207                True
     1537       210                True
     1538       210                True
     1669       260                True
     1666       260                True
     1667       260                True
     1668       260                True
      325       274                True
      878       274                True
     1061       293                True
     1260       293                True
     1290       393                True
      744       393                True
     1849       735                True
     1848       735                True
     1840       739                True
     1838       739                True
     1597   

### Inspect 2020 lakes that were assigned to multiple 2000 lakes

In [17]:
# Inspect 2020 lakes that were assigned to multiple 2000 lakes

duplicate_2020 = reliable_chain[
    reliable_chain['idx_2020'].duplicated(keep=False)
].sort_values('idx_2020')

print("Duplicate 2020 assignments:")
print(
    duplicate_2020[
        ['idx_1990', 'idx_2000', 'idx_2020',
         'reliable_1990_2000', 'reliable_2000_2020']
    ].to_string(index=False)
)

print("\nNumber of duplicated 2020 lakes:",
      duplicate_2020['idx_2020'].nunique())

Duplicate 2020 assignments:
 idx_1990  idx_2000  idx_2020  reliable_1990_2000  reliable_2000_2020
     1800      1732       200                True                True
     1801      1732       200                True                True
      324      1699       202                True                True
     1792      1699       202                True                True
     1796      1700       203                True                True
     1797      1700       203                True                True
     1794      1700       203                True                True
     1795      1700       203                True                True
     1581      1754       493                True                True
     1582      1754       493                True                True
      873       978       835                True                True
     2129       978       835                True                True
     2133       979       836                True             

### Fix duplicate matches: enforce one-to-one matching for 1990 → 2000

In [18]:
# Create candidate matches for 1990 → 2000
match_1990_2000 = pd.DataFrame({
    'idx_1990': np.arange(len(gdf_1990)),
    'idx_2000': indices_1990_2000.flatten(),
    'distance_km': distance_km_1990_2000
})

# Keep only reliable matches
match_1990_2000 = match_1990_2000[
    match_1990_2000['distance_km'] <= 1.0
].copy()

# Sort by distance so the closest match gets priority
match_1990_2000 = match_1990_2000.sort_values('distance_km')

# Enforce one-to-one matching
match_1990_2000_unique = match_1990_2000.drop_duplicates(
    subset='idx_2000',
    keep='first'
).copy()

print("Reliable candidate matches:", len(match_1990_2000))
print("Unique 1990 → 2000 matches:", len(match_1990_2000_unique))
print("2000 lakes used more than once:", match_1990_2000_unique['idx_2000'].duplicated().sum())

Reliable candidate matches: 2029
Unique 1990 → 2000 matches: 1939
2000 lakes used more than once: 0


### Fix duplicate matches: enforce one-to-one matching for 2000 → 2020

In [19]:
# Create candidate matches for 2000 → 2020
match_2000_2020 = pd.DataFrame({
    'idx_2000': np.arange(len(gdf_2000)),
    'idx_2020': indices_2000_2020.flatten(),
    'distance_km': distance_km_2000_2020
})

# Keep reliable matches only
match_2000_2020 = match_2000_2020[
    match_2000_2020['distance_km'] <= 1.0
].copy()

# Closest matches get priority
match_2000_2020 = match_2000_2020.sort_values('distance_km')

# Enforce one-to-one matching
match_2000_2020_unique = match_2000_2020.drop_duplicates(
    subset='idx_2020',
    keep='first'
).copy()

print("Reliable candidate matches:", len(match_2000_2020))
print("Unique 2000 → 2020 matches:", len(match_2000_2020_unique))
print("2020 lakes used more than once:", match_2000_2020_unique['idx_2020'].duplicated().sum())

Reliable candidate matches: 2075
Unique 2000 → 2020 matches: 1976
2020 lakes used more than once: 0


In [20]:
# Build the final one-to-one historical chain

chain_unique = match_1990_2000_unique[
    ['idx_1990', 'idx_2000', 'distance_km']
].copy()

chain_unique = chain_unique.rename(
    columns={'distance_km': 'distance_1990_2000_km'}
)

# Attach the unique 2000 → 2020 match
chain_unique = chain_unique.merge(
    match_2000_2020_unique[
        ['idx_2000', 'idx_2020', 'distance_km']
    ],
    on='idx_2000',
    how='inner'
)

chain_unique = chain_unique.rename(
    columns={'distance_km': 'distance_2000_2020_km'}
)

print("Final 1990 → 2000 → 2020 chains:", len(chain_unique))

print(
    "1990 lakes used once:",
    chain_unique['idx_1990'].is_unique
)

print(
    "2000 lakes used once:",
    chain_unique['idx_2000'].is_unique
)

print(
    "2020 lakes used once:",
    chain_unique['idx_2020'].is_unique
)

Final 1990 → 2000 → 2020 chains: 1864
1990 lakes used once: True
2000 lakes used once: True
2020 lakes used once: True


### Validating The Actual Distances

In [21]:
print("\n1990 → 2000 distance:")
print(chain_unique['distance_1990_2000_km'].describe())

print("\n2000 → 2020 distance:")
print(chain_unique['distance_2000_2020_km'].describe())

print("\nChains with both distances <= 0.5 km:",
      (
          (chain_unique['distance_1990_2000_km'] <= 0.5) &
          (chain_unique['distance_2000_2020_km'] <= 0.5)
      ).sum()
)

print("\nChains with either distance > 0.5 km:",
      (
          (chain_unique['distance_1990_2000_km'] > 0.5) |
          (chain_unique['distance_2000_2020_km'] > 0.5)
      ).sum()
)


1990 → 2000 distance:
count    1864.000000
mean        0.009995
std         0.057528
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.845908
Name: distance_1990_2000_km, dtype: float64

2000 → 2020 distance:
count    1864.000000
mean        0.012755
std         0.074875
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.996421
Name: distance_2000_2020_km, dtype: float64

Chains with both distances <= 0.5 km: 1847

Chains with either distance > 0.5 km: 17


### Calculate the growth feature from the 1,864 chains

In [22]:
# Pull area values for the final one-to-one historical chains

chain_unique['area_1990'] = (
    gdf_1990.iloc[chain_unique['idx_1990']]['area_km2'].values
)

chain_unique['area_2000'] = (
    gdf_2000.iloc[chain_unique['idx_2000']]['area_km2'].values
)

chain_unique['area_2020'] = (
    gdf_2020.iloc[chain_unique['idx_2020']]['area_km2'].values
)

# Carry the 2020 coordinates for the later merge
chain_unique['latitude'] = (
    gdf_2020.iloc[chain_unique['idx_2020']]['latitude_decimal'].values
)

chain_unique['longitude'] = (
    gdf_2020.iloc[chain_unique['idx_2020']]['longitude_decimal'].values
)

# Calculate total area change
chain_unique['area_change_km2'] = (
    chain_unique['area_2020'] -
    chain_unique['area_1990']
)

# Average annual area change over 30 years
chain_unique['annual_area_change_km2_per_year'] = (
    chain_unique['area_change_km2'] / 30
)

print("Final historical growth dataset:", len(chain_unique))

print("\nAnnual area change summary:")
print(
    chain_unique[
        ['area_1990',
         'area_2000',
         'area_2020',
         'area_change_km2',
         'annual_area_change_km2_per_year']
    ].describe()
)

Final historical growth dataset: 1864

Annual area change summary:
         area_1990    area_2000    area_2020  area_change_km2  \
count  1864.000000  1864.000000  1864.000000      1864.000000   
mean      0.043417     0.043617     0.043670         0.000253   
std       0.148363     0.149088     0.148652         0.012525   
min       0.004500     0.004500     0.004500        -0.280800   
25%       0.009900     0.009900     0.009900         0.000000   
50%       0.018900     0.018900     0.018900         0.000000   
75%       0.038025     0.038700     0.038700         0.000000   
max       4.860000     4.860000     4.860000         0.293400   

       annual_area_change_km2_per_year  
count                      1864.000000  
mean                          0.000008  
std                           0.000417  
min                          -0.009360  
25%                           0.000000  
50%                           0.000000  
75%                           0.000000  
max                

## Save final validated historical growth dataset

In [23]:
growth_data = chain_unique[[
    'latitude', 'longitude',
    'area_1990', 'area_2000', 'area_2020',
    'area_change_km2',
    'annual_area_change_km2_per_year'
]].copy()

growth_data.to_csv(
    '../data/processed/cpec_growth_rate_clean.csv',
    index=False
)

print("Saved. Shape:", growth_data.shape)

Saved. Shape: (1864, 7)


### Test overlap: match validated growth data to main GLOFGuard training dataset

In [24]:
main_lakes = pd.read_csv(
    '../data/processed/glofguard_lakes_v2_settlement.csv'
)

main_coords = np.radians(
    main_lakes[['latitude', 'longitude']].values
)

growth_coords = np.radians(
    growth_data[['latitude', 'longitude']].values
)

growth_tree = BallTree(
    growth_coords,
    metric='haversine'
)

main_distances, main_indices = growth_tree.query(
    main_coords,
    k=1
)

main_distance_km = main_distances.flatten() * 6371

GROWTH_MATCH_THRESHOLD_KM = 1.0

has_reliable_growth_match = (
    main_distance_km <= GROWTH_MATCH_THRESHOLD_KM
)

print("Total lakes in main dataset:", len(main_lakes))

print(
    f"Lakes with reliable growth match "
    f"(≤{GROWTH_MATCH_THRESHOLD_KM} km):",
    has_reliable_growth_match.sum()
)

print(
    f"Percentage with growth data: "
    f"{has_reliable_growth_match.mean()*100:.1f}%"
)

print("\nDistance distribution to nearest growth-rate lake:")

print(
    pd.Series(main_distance_km).describe()
)

Total lakes in main dataset: 8806
Lakes with reliable growth match (≤1.0 km): 3195
Percentage with growth data: 36.3%

Distance distribution to nearest growth-rate lake:
count    8806.000000
mean       10.799607
std        21.730007
min         0.000604
25%         0.396290
50%         2.215926
75%        11.224613
max       127.603145
dtype: float64


### Merge growth rate into main dataset, only for reliable matches (≤1 km)

In [25]:
# Create the growth feature with missing values by default
main_lakes['annual_area_change_km2_per_year'] = np.nan

# Assign growth only to reliable matches (<= 1 km)
reliable_mask = main_distance_km <= 1.0

main_lakes.loc[
    reliable_mask,
    'annual_area_change_km2_per_year'
] = growth_data.iloc[
    main_indices[reliable_mask].flatten()
]['annual_area_change_km2_per_year'].values

print("Total lakes:", len(main_lakes))

print(
    "Lakes with growth data:",
    main_lakes['annual_area_change_km2_per_year'].notna().sum()
)

print(
    "Lakes without growth data:",
    main_lakes['annual_area_change_km2_per_year'].isna().sum()
)

print(
    "Coverage:",
    main_lakes['annual_area_change_km2_per_year'].notna().mean() * 100
)

print("\nGrowth feature summary:")
print(
    main_lakes[
        'annual_area_change_km2_per_year'
    ].describe()
)

Total lakes: 8806
Lakes with growth data: 3195
Lakes without growth data: 5611
Coverage: 36.282080399727455

Growth feature summary:
count    3195.000000
mean       -0.000019
std         0.000444
min        -0.009360
25%         0.000000
50%         0.000000
75%         0.000000
max         0.009780
Name: annual_area_change_km2_per_year, dtype: float64


### Save as v3 - growth rate added

In [26]:
main_lakes.to_csv(
    '../data/processed/glofguard_lakes_v3_growth.csv',
    index=False
)

print("Saved: glofguard_lakes_v3_growth.csv")

Saved: glofguard_lakes_v3_growth.csv


### Audit v3: load fresh and verify structure, nulls, duplicates

In [27]:

df = pd.read_csv('../data/processed/glofguard_lakes_v3_growth.csv')

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (8806, 12)

Columns:
['sample_id', 'area', 'longitude', 'latitude', 'is_top200', 'temperature', 'rainfall', 'label', 'elevation', 'distance_to_nearest_settlement_km', 'nearest_settlement_name', 'annual_area_change_km2_per_year']

Missing values:
sample_id                               0
area                                    0
longitude                               0
latitude                                0
is_top200                               0
temperature                             0
rainfall                                0
label                                   0
elevation                               0
distance_to_nearest_settlement_km       0
nearest_settlement_name               817
annual_area_change_km2_per_year      5611
dtype: int64

Duplicate rows: 0


### Numeric summary check

In [28]:
print("\nNumeric summary:")
print(df.describe().T)


Numeric summary:
                                    count         mean          std  \
sample_id                          8806.0  4403.500000  2542.217569   
area                               8806.0     0.015485     0.079300   
longitude                          8806.0    74.735997     1.389159   
latitude                           8806.0    35.744941     0.570838   
temperature                        8806.0    -3.260794     4.077695   
rainfall                           8806.0     0.905964     0.390373   
label                              8806.0     0.039859     0.195639   
elevation                          8806.0  4187.092437   434.591369   
distance_to_nearest_settlement_km  8806.0    17.177632    10.260394   
annual_area_change_km2_per_year    3195.0    -0.000019     0.000444   

                                           min          25%          50%  \
sample_id                             1.000000  2202.250000  4403.500000   
area                                  0.000009  

### Geographic and physical validity checks

In [29]:
print("\nKey feature checks:")

print("Latitude outside Pakistan-region range:",
      ((df['latitude'] < 23) | (df['latitude'] > 38)).sum())

print("Longitude outside Pakistan-region range:",
      ((df['longitude'] < 60) | (df['longitude'] > 78)).sum())

print("Negative elevation:", (df['elevation'] < 0).sum())

print("Negative settlement distance:",
      (df['distance_to_nearest_settlement_km'] < 0).sum())

print("Missing growth values:",
      df['annual_area_change_km2_per_year'].isna().sum())


Key feature checks:
Latitude outside Pakistan-region range: 0
Longitude outside Pakistan-region range: 0
Negative elevation: 0
Negative settlement distance: 0
Missing growth values: 5611


### Create final historical growth dataset from validated chains

In [30]:
growth_data = chain_unique[
    [
        'latitude',
        'longitude',
        'area_1990',
        'area_2000',
        'area_2020',
        'area_change_km2',
        'annual_area_change_km2_per_year'
    ]
].copy()

print("Final historical growth dataset:", growth_data.shape)

print("\nGrowth feature summary:")
print(
    growth_data[
        ['annual_area_change_km2_per_year']
    ].describe()
)

Final historical growth dataset: (1864, 7)

Growth feature summary:
       annual_area_change_km2_per_year
count                      1864.000000
mean                          0.000008
std                           0.000417
min                          -0.009360
25%                           0.000000
50%                           0.000000
75%                           0.000000
max                           0.009780


### Save validated historical growth data, cpec_growth_rate_clean.csv

In [31]:
growth_data.to_csv(
    '../data/processed/cpec_growth_rate_clean.csv',
    index=False
)

print(
    "Saved: ../data/processed/cpec_growth_rate_clean.csv"
)

Saved: ../data/processed/cpec_growth_rate_clean.csv
